# Exercício 1: prevê o resultado

In [1]:
import pandas as pd

precos = pd.DataFrame({"produto": ["Sumo", "Água", "Maçã"],
    "preco": [1.0, 0.5, 1.5],
    "stock": [10, 25, 8]})

print(precos.shape)
print(precos["preco"].mean())
print(precos["stock"].sum())
print(precos["preco"] * 2)
print(precos["stock"].max())


(3, 3)
1.0
43
0    2.0
1    1.0
2    3.0
Name: preco, dtype: float64
25


# Exercício 2: completa a seleção de dados

In [5]:
import pandas as pd

df = pd.read_csv("cantina.csv")
muitas = df[df["quantidade"] > 60]
fruta = df[df["categoria"] == "Fruta"]
semana4 = df.loc[df["semana"] == 4, ["produto","preco"]]
print(len(muitas), len(fruta))
print(semana4)

5 4
              produto  preco
19   Sandes de queijo    1.5
20  Sandes de fiambre    1.6
21    Sumo de laranja    1.0
22               Água    0.6
23               Maçã    0.5
24            Iogurte    0.9


# Exercício 3: completa a análise

In [7]:
import pandas as pd

df = pd.read_csv("cantina.csv")
df = df.drop_duplicates()
df = df.dropna(subset=["quantidade"])
df["quantidade"] = df["quantidade"].astype(int)
total = df.groupby("produto")["quantidade"].sum()
print(total.sort_values(ascending=False).head(3))


produto
Água                294
Sumo de laranja     199
Sandes de queijo    174
Name: quantidade, dtype: int64


# Atividade: o relatório da cantina

In [10]:
import pandas as pd
import sqlite3

df = pd.read_csv("cantina.csv")
df = df.drop_duplicates()
df = df.dropna(subset=["quantidade"])
df["quantidade"] = df["quantidade"].astype(int)
df["receita"] = df["preco"] * df["quantidade"]

por_semana = df.groupby("semana")["receita"].sum().round(2)
print(por_semana)
por_categoria = df.groupby("categoria")["receita"].sum()
print(por_categoria.sort_values(ascending=False).head(1))
por_semana.to_csv("receita_semana.csv")

ligacao = sqlite3.connect("cantina.db", timeout=5)
try:
    df.to_sql("vendas", ligacao, if_exists="replace", index=False)
    q = "SELECT SUM(quantidade) AS total FROM vendas WHERE semana = ?"
    print(pd.read_sql_query(q, ligacao, params=[4]))
finally:
    ligacao.close()

semana
1    258.3
2    197.3
3    260.6
4    266.6
Name: receita, dtype: float64
categoria
Comida    445.0
Name: receita, dtype: float64
   total
0    272


# Desafio 1: a variação da receita de semana para semana

In [11]:
import pandas as pd

df = pd.read_csv("cantina.csv").drop_duplicates().dropna(subset=["quantidade"])
df["receita"] = df["preco"] * df["quantidade"]
por_semana = df.groupby("semana")["receita"].sum()
variacao = (por_semana.pct_change() * 100).round(1)
print(variacao)
print(variacao.idxmin())

semana
1     NaN
2   -23.6
3    32.1
4     2.3
Name: receita, dtype: float64
2


# Desafio 2: SQL ou Pandas, a mesma resposta

In [15]:
import sqlite3
import pandas as pd

df = pd.read_csv("cantina.csv").drop_duplicates().dropna(subset=["quantidade"])
df["quantidade"] = df["quantidade"].astype(int)
ligacao = sqlite3.connect("cantina.db", timeout=5)
try:
    df.to_sql("vendas", ligacao, if_exists="replace", index=False)
    consulta = ("SELECT produto, SUM(quantidade) AS vendas FROM vendas "
                "WHERE semana >= ? GROUP BY produto "
                "HAVING SUM(quantidade) > ? ORDER BY produto")
    sql = pd.read_sql_query(consulta, ligacao, params=[3, 70])
finally:
    ligacao.close()
    print(sql)

filtrado = df[df["semana"] >= 3]
soma = filtrado.groupby("produto")["quantidade"].sum().reset_index(name="vendas")
com_pandas = soma[soma["vendas"] > 70].sort_values("produto").reset_index(drop=True)
print(com_pandas)
print(sql.equals(com_pandas))

             produto  vendas
0  Sandes de fiambre      77
1   Sandes de queijo      87
2    Sumo de laranja      96
3               Água     158
             produto  vendas
0  Sandes de fiambre      77
1   Sandes de queijo      87
2    Sumo de laranja      96
3               Água     158
True
